# Decision Tree: Adult Income classification

This notebook uses the Adult Income dataset from the `models.md` table to predict whether income exceeds $50K.
It practices encoding mixed categorical/numerical features and explores entropy/Gini splitting with pruning.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# ---------- Load ----------
cols = ['age', 'workclass', 'fnlwgt', 'education', 'education_num',
        'marital_status', 'occupation', 'relationship', 'race', 'sex',
        'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
        'income']
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
df = pd.read_csv(url, header=None, names=cols, skipinitialspace=True)

# ---------- Clean ----------
# Remove rows with missing values marked as '?'.
df = df.replace('?', pd.NA).dropna()
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

X = df.drop('income', axis=1)
y = df['income']

# ---------- Split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------- Encode ----------
# Fit encoding on training data only to avoid leakage.
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include='number').columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
])
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)

# ---------- Decision Tree ----------
model = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
model.fit(X_train_enc, y_train)
y_pred = model.predict(X_test_enc)

print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1 score:  {f1_score(y_test, y_pred):.4f}')
print('\nClassification report:\n',
      classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['<=50K', '>50K']).plot(cmap='Blues')
plt.title('Decision Tree (entropy, max_depth=5): confusion matrix')
plt.show()

In [ ]:
import numpy as np
# Feature Importances
importances = model.feature_importances_
try:
    feature_names = preprocessor.get_feature_names_out()
except:
    feature_names = np.array([f"Feature {i}" for i in range(len(importances))])
    
indices = np.argsort(importances)[-15:] # Top 15

plt.figure(figsize=(10, 6))
plt.title("Top 15 Feature Importances - Decision Tree")
plt.barh(range(len(indices)), importances[indices], align="center", color='teal')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()


## Decision Tree: splitting criteria, pruning, and evaluation

### Notation

- $S$: set of samples at a node.
- $c \in \{0, 1\}$: class ($0$ = ≤50K, $1$ = >50K).
- $p_c = |S_c| / |S|$: proportion of class $c$ in $S$.
- $A$: a candidate feature to split on; $S_v$: subset where feature $A$ takes value $v$.

### Entropy

Entropy measures impurity — the uncertainty about the class label:

$$H(S) = -\sum_{c} p_c \log_2 p_c$$

$H = 0$ when all samples belong to one class (pure). $H = 1$ for a binary split with equal proportions.

### Gini impurity

An alternative impurity measure:

$$G(S) = 1 - \sum_{c} p_c^2$$

Gini and entropy usually produce similar trees. Gini is slightly faster (no logarithm) and is scikit-learn's default.

### Information Gain

The tree selects the split that maximizes information gain — the reduction in impurity:

$$\mathrm{IG}(S, A) = H(S) - \sum_{v} \frac{|S_v|}{|S|}\, H(S_v)$$

**Maximize** IG: a higher value means the split produces purer child nodes.

### Gain Ratio

IG is biased toward features with many distinct values. Gain ratio corrects for this:

$$\mathrm{GainRatio}(S, A) = \frac{\mathrm{IG}(S, A)}{\mathrm{SplitInfo}(S, A)}, \qquad \mathrm{SplitInfo} = -\sum_{v} \frac{|S_v|}{|S|}\log_2 \frac{|S_v|}{|S|}$$

### Pruning

Unpruned trees tend to overfit. Two strategies control tree complexity:

**Pre-pruning** (stop growing early):
- `max_depth`: maximum tree depth.
- `min_samples_split`: minimum samples to attempt a split.
- `min_samples_leaf`: minimum samples in any leaf.

**Post-pruning** (grow full, then prune):
- `ccp_alpha` (cost-complexity pruning): higher $\alpha$ removes more nodes; select via cross-validation.

### Key hyperparameters

| Parameter | Typical values | Effect |
| --- | --- | --- |
| `criterion` | `'gini'`, `'entropy'`, `'log_loss'` | Impurity function used for splits |
| `max_depth` | 3–20 or `None` | Limits tree depth; primary pre-pruning control |
| `min_samples_split` | 2–20 | Minimum samples required to split an internal node |
| `min_samples_leaf` | 1–10 | Minimum samples in each leaf |
| `ccp_alpha` | 0.0–0.05 | Cost-complexity pruning parameter; higher = simpler tree |

### Test-set evaluation

$$\mathrm{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}$$

$$\mathrm{Precision} = \frac{TP}{TP+FP}, \qquad \mathrm{Recall} = \frac{TP}{TP+FN}$$

$$F_1 = 2\cdot\frac{\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}$$

**Maximize** Accuracy, Precision, Recall, and $F_1$: each ranges from $0$ (worst) to $1$ (best). For income prediction, >50K recall measures how many high earners the model catches, and >50K precision measures how often a predicted high earner truly earns >50K.

## Decision Tree from scratch (NumPy only)

The implementation below mirrors the from-scratch pattern used in the Logistic Regression and
Softmax Regression notebooks. It uses only NumPy (plus `urllib`/`csv` for downloading) for:
stratified train/test split, manual one-hot and standard-scaling, recursive tree building
with entropy-based information gain, and hand-computed evaluation metrics.

The tree handles both numerical splits (threshold) and categorical splits (one-hot columns),
supports `max_depth` pre-pruning, and uses the same Adult Income dataset.

In [ ]:
import numpy as np
import csv
import urllib.request
import io

# ── Load data (no pandas) ────────────────────────────────────────────
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
response = urllib.request.urlopen(url)
raw = response.read().decode('utf-8')
reader = csv.reader(io.StringIO(raw), skipinitialspace=True)

col_names = ['age', 'workclass', 'fnlwgt', 'education', 'education_num',
             'marital_status', 'occupation', 'relationship', 'race', 'sex',
             'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
             'income']
num_indices = [0, 2, 4, 10, 11, 12]          # indices of numerical columns
cat_indices = [1, 3, 5, 6, 7, 8, 9, 13]      # indices of categorical columns

rows = [r for r in reader if len(r) == len(col_names) and '?' not in r]

# Separate label
labels = np.array([1 if r[14].strip() == '>50K' else 0 for r in rows])

# Build numerical matrix
num_data = np.array([[float(r[i]) for i in num_indices] for r in rows])

# Build one-hot for each categorical column
cat_raw = [[r[i] for i in cat_indices] for r in rows]
# Collect unique values per categorical column (from training set later)
# For now store raw strings; we'll encode after splitting.

print(f'Loaded {len(rows)} rows, {len(num_indices)} numerical + {len(cat_indices)} categorical features')

In [ ]:
# ── Stratified train/test split (pure NumPy) ─────────────────────────
def stratified_split_numpy(y, test_size=0.2, seed=42):
    np.random.seed(seed)
    train_idx, test_idx = [], []
    for label_value in np.unique(y):
        class_indices = np.where(y == label_value)[0]
        shuffled = np.random.permutation(class_indices)
        n_test = int(len(shuffled) * test_size)
        test_idx.extend(shuffled[:n_test])
        train_idx.extend(shuffled[n_test:])
    return np.array(train_idx), np.array(test_idx)

train_idx, test_idx = stratified_split_numpy(labels, test_size=0.2, seed=42)

num_train, num_test = num_data[train_idx], num_data[test_idx]
cat_train = [cat_raw[i] for i in train_idx]
cat_test  = [cat_raw[i] for i in test_idx]
y_train, y_test = labels[train_idx], labels[test_idx]

# ── Manual standard scaling for numerical features (fit on train) ────
def fit_scaler(X):
    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std[std == 0] = 1.0
    return mean, std

def apply_scaler(X, mean, std):
    return (X - mean) / std

mean, std = fit_scaler(num_train)
num_train_s = apply_scaler(num_train, mean, std)
num_test_s  = apply_scaler(num_test, mean, std)

# ── Manual one-hot encoding for categorical features (fit on train) ──
n_cat_cols = len(cat_indices)
# Learn unique values per column from training data
cat_vocab = []
for col in range(n_cat_cols):
    unique_vals = sorted(set(row[col] for row in cat_train))
    cat_vocab.append(unique_vals)

def one_hot_encode(cat_rows, vocab):
    """Encode list-of-lists of categorical strings into a 2D NumPy array."""
    parts = []
    for col, unique_vals in enumerate(vocab):
        val_to_idx = {v: i for i, v in enumerate(unique_vals)}
        n_vals = len(unique_vals)
        block = np.zeros((len(cat_rows), n_vals))
        for row_i, row in enumerate(cat_rows):
            idx = val_to_idx.get(row[col])
            if idx is not None:
                block[row_i, idx] = 1.0
        parts.append(block)
    return np.hstack(parts)

cat_train_enc = one_hot_encode(cat_train, cat_vocab)
cat_test_enc  = one_hot_encode(cat_test, cat_vocab)

# ── Combine numerical + categorical ──────────────────────────────────
X_train = np.hstack([num_train_s, cat_train_enc])
X_test  = np.hstack([num_test_s, cat_test_enc])

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_train: {y_train.shape} (pos rate {y_train.mean():.3f})')

In [ ]:
# ── Decision Tree from scratch ───────────────────────────────────────

def entropy(y):
    """H(S) = -sum p_c * log2(p_c)"""
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y)
    probs = counts[counts > 0] / len(y)
    return -np.sum(probs * np.log2(probs))

def information_gain(y, left_mask):
    """IG(S, split) = H(S) - weighted avg of children entropy."""
    right_mask = ~left_mask
    n = len(y)
    n_left, n_right = left_mask.sum(), right_mask.sum()
    if n_left == 0 or n_right == 0:
        return 0.0
    parent_ent = entropy(y)
    child_ent = (n_left / n) * entropy(y[left_mask]) + (n_right / n) * entropy(y[right_mask])
    return parent_ent - child_ent

def best_split(X, y):
    """
    Find the feature and threshold that maximizes information gain.
    For each feature, try all unique midpoints as thresholds (left: <= threshold).
    """
    best_ig = 0.0
    best_feat = None
    best_thresh = None
    n_samples, n_features = X.shape

    for feat in range(n_features):
        col = X[:, feat]
        unique_vals = np.unique(col)
        if len(unique_vals) <= 1:
            continue

        # For binary one-hot columns (0/1), only one threshold: 0.5
        if len(unique_vals) == 2 and set(unique_vals).issubset({0.0, 1.0}):
            thresholds = [0.5]
        else:
            # Midpoints between sorted unique values
            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2

        for thresh in thresholds:
            left_mask = col <= thresh
            ig = information_gain(y, left_mask)
            if ig > best_ig:
                best_ig = ig
                best_feat = feat
                best_thresh = thresh

    return best_feat, best_thresh, best_ig

class DecisionNode:
    """Internal node: stores feature index and threshold."""
    def __init__(self, feature, threshold, left, right):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right

class LeafNode:
    """Leaf node: stores the majority class."""
    def __init__(self, y):
        counts = np.bincount(y)
        self.prediction = np.argmax(counts)

def build_tree(X, y, depth=0, max_depth=5, min_samples_split=2):
    """
    Recursively build the decision tree.

    Stopping criteria:
    - All samples belong to one class (pure node).
    - max_depth reached.
    - Fewer samples than min_samples_split.
    - No information gain from any split.
    """
    # Stopping conditions
    if len(np.unique(y)) == 1:
        return LeafNode(y)
    if depth >= max_depth:
        return LeafNode(y)
    if len(y) < min_samples_split:
        return LeafNode(y)

    feat, thresh, ig = best_split(X, y)
    if feat is None or ig == 0:
        return LeafNode(y)

    left_mask = X[:, feat] <= thresh
    right_mask = ~left_mask

    left_child  = build_tree(X[left_mask], y[left_mask], depth + 1, max_depth, min_samples_split)
    right_child = build_tree(X[right_mask], y[right_mask], depth + 1, max_depth, min_samples_split)

    return DecisionNode(feat, thresh, left_child, right_child)

def predict_one(node, x):
    """Traverse the tree to predict a single sample."""
    if isinstance(node, LeafNode):
        return node.prediction
    if x[node.feature] <= node.threshold:
        return predict_one(node.left, x)
    else:
        return predict_one(node.right, x)

def predict_tree(node, X):
    """Predict for all samples in X."""
    return np.array([predict_one(node, x) for x in X])

def count_nodes(node):
    """Count total nodes (internal + leaves) in the tree."""
    if isinstance(node, LeafNode):
        return 1
    return 1 + count_nodes(node.left) + count_nodes(node.right)

# ── Train ────────────────────────────────────────────────────────────
print('Building tree (entropy, max_depth=5)...')
tree = build_tree(X_train, y_train, max_depth=5, min_samples_split=2)
print(f'Tree has {count_nodes(tree)} nodes')

# ── Predict ──────────────────────────────────────────────────────────
y_pred = predict_tree(tree, X_test)

# ── Manual metrics ───────────────────────────────────────────────────
tp = np.sum((y_pred == 1) & (y_test == 1))
tn = np.sum((y_pred == 0) & (y_test == 0))
fp = np.sum((y_pred == 1) & (y_test == 0))
fn = np.sum((y_pred == 0) & (y_test == 1))

accuracy  = (tp + tn) / len(y_test)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f'\nAccuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')
print(f'\nConfusion Matrix:')
print(f'  [[TN={tn}  FP={fp}]')
print(f'   [FN={fn}  TP={tp}]]')